In [26]:
from google.colab import files
files.upload()

Saving warehouse_stock_movements2.csv to warehouse_stock_movements2.csv


{'warehouse_stock_movements2.csv': b'movement_id,warehouse_id,product_id,product_name,movement_type,date_movement,quantity_changed,reorder_level\r\n9001,1001,10,Quantum Laptop Pro,IN,2026-06-01,50,15\r\n9002,1001,10,Quantum Laptop Pro,OUT,2026-06-05,-15,15\r\n9003,1002,10,Quantum Laptop Pro,IN,2026-06-02,20,15\r\n9004,1002,10,Quantum Laptop Pro,OUT,2026-06-14,-14,15\r\n9005,1001,20,Wireless Ergonomic Mouse,IN,,200,50\r\n9006,1001,20,Wireless Ergonomic Mouse,OUT,2026-06-12,-220,50\r\n9007,1003,20,Wireless Ergonomic Mouse,IN,2026-06-04,450,50\r\n9008,1003,20,Wireless Ergonomic Mouse,OUT,2026-06-10,-50,50\r\n9009,1002,30,Running Shoes Cloud-9,IN,2026-06-08,120,30\r\n9010,1002,30,Running Shoes Cloud-9,OUT,2026-06-14,-95,30\r\n9011,1003,30,Running Shoes Cloud-9,IN,2026-06-09,40,30\r\n9012,1003,30,Running Shoes Cloud-9,OUT,2026-06-15,-35,30\r\n9013,1001,40,Stainless Water Bottle,IN,2026-06-11,300,75\r\n9014,1001,40,Stainless Water Bottle,OUT,2026-06-13,10,75\r\n9015,1002,40,Stainless Water B

In [3]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('practice').getOrCreate()

In [27]:
warehouse_stock=spark.read.csv("warehouse_stock_movements2.csv",header=True,inferSchema=True)
warehouse_stock.show()

+-----------+------------+----------+--------------------+-------------+-------------+----------------+-------------+
|movement_id|warehouse_id|product_id|        product_name|movement_type|date_movement|quantity_changed|reorder_level|
+-----------+------------+----------+--------------------+-------------+-------------+----------------+-------------+
|       9001|        1001|        10|  Quantum Laptop Pro|           IN|   2026-06-01|              50|           15|
|       9002|        1001|        10|  Quantum Laptop Pro|          OUT|   2026-06-05|             -15|           15|
|       9003|        1002|        10|  Quantum Laptop Pro|           IN|   2026-06-02|              20|           15|
|       9004|        1002|        10|  Quantum Laptop Pro|          OUT|   2026-06-14|             -14|           15|
|       9005|        1001|        20|Wireless Ergonomi...|           IN|         NULL|             200|           50|
|       9006|        1001|        20|Wireless Ergonomi..

In [28]:
from pyspark.sql.functions import sum
warehouse_quantity=warehouse_stock.groupBy("warehouse_id").agg(sum("quantity_changed").alias("current_stock"))
warehouse_quantity.show()

+------------+-------------+
|warehouse_id|current_stock|
+------------+-------------+
|        1002|          116|
|        1001|          325|
|        1003|          475|
+------------+-------------+



In [33]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col,when

warehouse_product_stock = warehouse_stock.groupBy("warehouse_id", "product_id", "product_name") \
    .agg(
        F.sum("quantity_changed").alias("current_stock"),
        F.first("reorder_level").alias("reorder_level")
    )

warehouse_product_stock = warehouse_product_stock.withColumn(
    "stock_status",
    when(col("current_stock") < col("reorder_level"), "Understock")
    .when(col("current_stock") > (col("reorder_level") * 3), "Overstock")
    .otherwise("Normal")
)

warehouse_product_stock.orderBy("warehouse_id", "product_id").show()


+------------+----------+--------------------+-------------+-------------+------------+
|warehouse_id|product_id|        product_name|current_stock|reorder_level|stock_status|
+------------+----------+--------------------+-------------+-------------+------------+
|        1001|        10|  Quantum Laptop Pro|           35|           15|      Normal|
|        1001|        20|Wireless Ergonomi...|          -20|           50|  Understock|
|        1001|        40|Stainless Water B...|          310|           75|   Overstock|
|        1002|        10|  Quantum Laptop Pro|            6|           15|  Understock|
|        1002|        30|Running Shoes Clo...|           25|           30|  Understock|
|        1002|        40|Stainless Water B...|           85|           75|      Normal|
|        1003|        20|Wireless Ergonomi...|          400|           50|   Overstock|
|        1003|        30|Running Shoes Clo...|            5|           30|  Understock|
|        1003|        50| Mechan

In [34]:
print("Displaying understocked warehouses")
warehouse_product_stock.filter(col("stock_status") == "Understock").show()

Displaying understocked warehouses
+------------+----------+--------------------+-------------+-------------+------------+
|warehouse_id|product_id|        product_name|current_stock|reorder_level|stock_status|
+------------+----------+--------------------+-------------+-------------+------------+
|        1001|        20|Wireless Ergonomi...|          -20|           50|  Understock|
|        1003|        30|Running Shoes Clo...|            5|           30|  Understock|
|        1002|        30|Running Shoes Clo...|           25|           30|  Understock|
|        1002|        10|  Quantum Laptop Pro|            6|           15|  Understock|
|        1003|        50| Mechanical Keyboard|           70|          120|  Understock|
+------------+----------+--------------------+-------------+-------------+------------+



In [35]:
print("Overstocked warehouses")
warehouse_product_stock.filter(col("stock_status") == "Overstock").show()

Overstocked warehouses
+------------+----------+--------------------+-------------+-------------+------------+
|warehouse_id|product_id|        product_name|current_stock|reorder_level|stock_status|
+------------+----------+--------------------+-------------+-------------+------------+
|        1003|        20|Wireless Ergonomi...|          400|           50|   Overstock|
|        1001|        40|Stainless Water B...|          310|           75|   Overstock|
+------------+----------+--------------------+-------------+-------------+------------+



In [36]:
warehouse_product_stock.write.csv("warehouse_stock_insights", header=True, mode="overwrite")